# DL Demonstration: загрузка PRD-модели из MLflow и инференс

Этот ноутбук **не обучает** модель. Он демонстрирует практическую работу с
уже залогированной в MLflow моделью — той самой, что получена в `DL_Experiments.ipynb`
(или через `train_cli.py`) и помечена тегом **PRD**.

## План

1. Подключение к MLflow tracking server
2. Поиск последнего run-а с тегом `stage = PRD`
3. Загрузка модели и метаинформации
4. Инференс на тестовом резюме
5. Визуализация распознанных сущностей

---
## 1. Зависимости и подключение к MLflow

Предполагается, что MLflow поднят локально через docker-compose (см. `mlflow/README.md`):
- **MLflow** — http://localhost:5001
- **MinIO (S3)** — http://localhost:9000

In [ ]:
!pip install mlflow boto3 spacy --quiet

In [ ]:
import os
import json
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient

# Configure MLflow + S3 endpoint
os.environ['MLFLOW_TRACKING_URI'] = 'http://localhost:5001'
os.environ['MLFLOW_S3_ENDPOINT_URL'] = 'http://localhost:9000'
os.environ['AWS_ACCESS_KEY_ID'] = 'minioadmin'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'minioadmin'

EXPERIMENT_NAME = 'resume-ner'
PROD_TAG = 'PRD'

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
client = MlflowClient()
print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experiment:          {EXPERIMENT_NAME}')
print(f'Looking for tag:     stage = {PROD_TAG}')

---
## 2. Поиск PRD-модели

In [ ]:
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise RuntimeError(
        f"Experiment '{EXPERIMENT_NAME}' not found. "
        f"Run DL_Experiments.ipynb or train_cli.py first.")

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.stage = '{PROD_TAG}'",
    order_by=['start_time DESC'],
    max_results=5,
)

if not runs:
    raise RuntimeError(f"No runs tagged '{PROD_TAG}' found in experiment.")

print(f'Found {len(runs)} run(s) tagged {PROD_TAG}:\n')
for i, r in enumerate(runs):
    f1 = r.data.metrics.get('test_entity_f1', float('nan'))
    print(f'  [{i}] {r.info.run_id}  Entity-F1={f1:.4f}  start={r.info.start_time}')

prd_run = runs[0]
print(f'\nUsing latest run: {prd_run.info.run_id}')

In [ ]:
# Параметры и метрики выбранного run-а
import pandas as pd

params_df = pd.DataFrame([{'param': k, 'value': v} for k, v in prd_run.data.params.items()])
metrics = prd_run.data.metrics

print('=== PARAMETERS ===')
print(params_df.to_string(index=False))

print('\n=== KEY METRICS ===')
for k in ['test_entity_f1', 'test_entity_precision', 'test_entity_recall',
          'test_token_accuracy', 'best_dev_f1', 'train_time_sec']:
    if k in metrics:
        print(f'  {k:30s} = {metrics[k]:.4f}')

print('\n=== TAGS ===')
for k, v in prd_run.data.tags.items():
    if not k.startswith('mlflow.'):
        print(f'  {k:25s} = {v}')

---
## 3. Загрузка модели

In [ ]:
model_uri = f'runs:/{prd_run.info.run_id}/model'
print(f'Loading model from: {model_uri}\n')

nlp = mlflow.spacy.load_model(model_uri)

print(f'Pipeline: {nlp.pipe_names}')
print(f'Labels ({len(nlp.get_pipe("ner").labels)}): '
      f'{list(nlp.get_pipe("ner").labels)}')

---
## 4. Тестовый предикт

Реальное резюме разработчика — не из обучающей выборки. Проверяем, как модель
извлекает сущности.

In [ ]:
SAMPLE_RESUME = '''
John Smith
Senior Software Engineer - Google
San Francisco, CA - Email me: john.smith@example.com

WORK EXPERIENCE

Senior Software Engineer
Google -
January 2020 to Present
Role: Building distributed systems for search infrastructure.

Software Engineer
Microsoft -
June 2017 to December 2019
Worked on Azure cloud platform, primarily on storage services.

EDUCATION

M.S. in Computer Science
Stanford University - Stanford, CA
2015 to 2017

B.S. in Computer Engineering
MIT - Cambridge, MA
2011 to 2015

SKILLS
Python, Java, Go, Kubernetes, Docker, AWS, GCP, PostgreSQL, MongoDB, Machine Learning,
Distributed Systems, Microservices, Linux
'''

doc = nlp(SAMPLE_RESUME)

print(f'Found {len(doc.ents)} entities:\n')
for ent in doc.ents:
    print(f'  [{ent.label_:22s}] {repr(ent.text)}  '
          f'({ent.start_char}..{ent.end_char})')

### Группировка по типам сущностей

In [ ]:
from collections import defaultdict

by_label = defaultdict(list)
for ent in doc.ents:
    by_label[ent.label_].append(ent.text)

result = {label: list(set(values)) for label, values in by_label.items()}
print(json.dumps(result, indent=2, ensure_ascii=False))

### Цветная визуализация через `spacy.displacy`

In [ ]:
from spacy import displacy

colors = {
    'Name': '#FF6B6B',
    'Email Address': '#4ECDC4',
    'Location': '#95E1D3',
    'College Name': '#FFE66D',
    'Degree': '#A8DADC',
    'Graduation Year': '#F4A261',
    'Companies worked at': '#E76F51',
    'Designation': '#9B5DE5',
    'Skills': '#F15BB5',
    'Years of Experience': '#00BBF9',
}
options = {'ents': list(nlp.get_pipe('ner').labels), 'colors': colors}

displacy.render(doc, style='ent', jupyter=True, options=options)

---
## 5. Инференс на реальном резюме из датасета (для контроля)

Берём одно резюме из тестовой выборки и сравниваем предсказание с эталонной разметкой.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent))
from data_utils import full_clean_pipeline, train_test_split

DATASET_PATH = Path.cwd().parent / 'datasets' / 'dataturks' / 'Entity Recognition in Resumes.json'
if DATASET_PATH.exists():
    data = full_clean_pipeline(DATASET_PATH)
    _, test_data = train_test_split(data, test_size=0.1, random_state=42)
    print(f'Test set: {len(test_data)} resumes')
    real_text, real_ann = test_data[0]
else:
    real_text, real_ann = None, None
    print('Dataset not found; skipping this section.')

In [ ]:
if real_text:
    real_doc = nlp(real_text)

    true_set = {(s, e, l) for s, e, l in real_ann['entities']}
    pred_set = {(ent.start_char, ent.end_char, ent.label_) for ent in real_doc.ents}

    matched = true_set & pred_set
    only_true = true_set - pred_set
    only_pred = pred_set - true_set

    print(f'Resume length: {len(real_text)} chars')
    print(f'True entities:        {len(true_set)}')
    print(f'Predicted entities:   {len(pred_set)}')
    print(f'Correct (TP):         {len(matched)}')
    print(f'Missed  (FN):         {len(only_true)}')
    print(f'Extra   (FP):         {len(only_pred)}')

    if matched:
        print('\nCorrect predictions:')
        for s, e, l in list(matched)[:10]:
            print(f'  [{l:22s}] {repr(real_text[s:e][:50])}')

In [ ]:
# Цветная визуализация реального резюме
if real_text:
    displacy.render(real_doc, style='ent', jupyter=True, options=options)

---
## Итог

Демонстрация работает следующим образом:

1. Подключаемся к MLflow tracking server (`http://localhost:5001`)
2. Находим **последний run** с тегом `stage = PRD`
3. Через `mlflow.spacy.load_model('runs:/<id>/model')` подтягиваем артефакт из MinIO (S3)
4. Применяем модель к произвольному тексту резюме — получаем именованные сущности

Никакого обучения здесь нет — только использование готового артефакта. Это и есть
production-сценарий: модель развёрнута, версионирована, воспроизводима.

### Для сравнения: то же самое через CLI (бонус)

```bash
python demo_cli.py
```